In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("popular_anime.csv")
print("Raw shape :", df.shape)

In [ ]:
# ---------- Inspect the missing values ----------
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "percent": (df.isna().sum() * 100 / len(df)).round(2)
})
print(missing[missing["missing"] > 0].sort_values("missing", ascending=False))

In [ ]:
# ---------- Drop the columns not required ----------
df = df.drop(columns=["image", "trailer", "synopsis"])
print("Shape after dropping columns :", df.shape)

# ---------- Remove records with essential fields missing ----------
df = df.dropna(subset=["name", "score", "type"])
print("Shape after dropping incomplete rows :", df.shape)

In [ ]:
# ---------- Fill the remaining missing values ----------
df["episodes"]  = df["episodes"].fillna(df["episodes"].median())
df["genres"]    = df["genres"].fillna("Unknown")
df["studios"]   = df["studios"].fillna("Unknown")
df["producers"] = df["producers"].fillna("Unknown")
df["rating"]    = df["rating"].fillna(df["rating"].mode()[0])

print("Remaining missing values :")
print(df.isna().sum()[df.isna().sum() > 0])

In [ ]:
# ---------- Detect and remove duplicates ----------
print("Duplicate rows        :", df.duplicated().sum())
print("Duplicate titles      :", df.duplicated(subset=["name"]).sum())

df = df.drop_duplicates(subset=["id"])
print("Shape after removing duplicates :", df.shape)

In [ ]:
# ---------- Convert the date columns to datetime ----------
df["aired_from"] = pd.to_datetime(df["aired_from"], errors="coerce", utc=True)
df["aired_to"]   = pd.to_datetime(df["aired_to"],   errors="coerce", utc=True)

print(df[["name", "aired_from", "aired_to"]].head(3))
print()
print("aired_from dtype :", df["aired_from"].dtype)

In [ ]:
# ---------- Extract numeric duration in minutes ----------
import re

def to_minutes(text):
    text = str(text)
    hrs  = re.search(r"(\d+)\s*hr", text)
    mins = re.search(r"(\d+)\s*min", text)
    secs = re.search(r"(\d+)\s*sec", text)
    total = 0
    if hrs:  total += int(hrs.group(1)) * 60
    if mins: total += int(mins.group(1))
    if secs: total += int(secs.group(1)) / 60
    return total if total > 0 else np.nan

df["duration_min"] = df["duration_per_ep"].apply(to_minutes)
df["duration_min"] = df["duration_min"].fillna(df["duration_min"].median())

print(df[["duration_per_ep", "duration_min"]].head(6))

In [ ]:
# ---------- Clean the text columns ----------
df["name"] = df["name"].str.strip()
df["type"] = df["type"].str.strip().str.upper()
df["status"] = df["status"].str.strip().str.title()

print("Distinct types  :", sorted(df["type"].unique()))
print("Distinct status :", sorted(df["status"].unique()))

In [ ]:
# ---------- Detect outliers using the IQR method ----------
q1, q3 = df["episodes"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print("Q1 =", q1, " Q3 =", q3, " IQR =", iqr)
print("Lower limit =", round(low, 2), " Upper limit =", round(high, 2))
print("Outliers detected :", ((df["episodes"] < low) | (df["episodes"] > high)).sum())

In [ ]:
# ---------- Treat the outliers by capping ----------
df["episodes_capped"] = df["episodes"].clip(lower=max(low, 1), upper=high)

print("Before capping -> max :", df["episodes"].max())
print("After  capping -> max :", df["episodes_capped"].max())
print(df[["episodes", "episodes_capped"]].describe().round(2))

In [ ]:
# ---------- Create derived columns ----------
df["release_year"] = df["aired_from"].dt.year
df["total_minutes"] = df["episodes"] * df["duration_min"]

print(df[["name", "release_year", "episodes", "duration_min", "total_minutes"]].head())

In [ ]:
# ---------- Bin the score into rating bands ----------
df["score_band"] = pd.cut(df["score"],
                          bins=[0, 5, 6.5, 7.5, 8.5, 10],
                          labels=["Poor", "Average", "Good", "Very Good", "Excellent"])

print(df["score_band"].value_counts().sort_index())

In [ ]:
# ---------- Encode the categorical columns ----------
df["type_code"] = df["type"].astype("category").cat.codes
encoded = pd.get_dummies(df["score_band"], prefix="band").astype(int)

print(df[["type", "type_code"]].drop_duplicates().sort_values("type_code").head())
print()
print(encoded.head())

In [ ]:
# ---------- Scale the numeric columns using min-max normalisation ----------
for col in ["score", "episodes_capped", "duration_min"]:
    lo, hi = df[col].min(), df[col].max()
    df[col + "_scaled"] = (df[col] - lo) / (hi - lo)

print(df[["score_scaled", "episodes_capped_scaled", "duration_min_scaled"]].describe().round(4))

In [ ]:
# ---------- Final cleaned dataset ----------
print("Cleaned shape        :", df.shape)
print("Missing values left  :", int(df[["name", "score", "type", "episodes"]].isna().sum().sum()))
print()
print(df[["name", "type", "episodes", "score", "score_band", "release_year"]].head(8).to_string(index=False))